# Sample and null permute portion of every conversation.

Requested by @RickDale

In [1]:
import os
import numpy as np
import pandas as pd
import re
import shutil

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
SERVER_READY_DATA = '../data/server_ready'

OUTPUT_FOLDER = '../data/server_ready_'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

## Iterate, permute

In [4]:
fs = grab_all_files(SERVER_READY_DATA)
len(fs)

26

### Everything other than CANDOR

In [5]:
NON_CANDOR = [f for f in fs if (not bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f)))]
len(NON_CANDOR)

26

In [6]:
families = set([f.split('/')[-1].split('-')[1].replace('.parquet', '') for f in NON_CANDOR])
families

{'callfriend', 'callhome', 'croatian', 'finchat_corpus', 'yiddish'}

In [7]:
for fam in tqdm(families):

    familiars = [f for f in fs if (fam in f)]
    df = [
        pq.ParquetFile(f).read(columns=['corpus', 'conversation_id', 'x_turn_id', 'y_turn_id', 'x_speaker', 'y_speaker', 'x_utterance', 'y_utterance']).to_pandas()
        for f in familiars
    ]

    df = pd.concat(df, ignore_index=True)

    good_y = ~df['y_utterance'].isna()

    if 'yiddish' in df['corpus'].unique():
        df['corpus'] = 'yiddish-yid'

    df['lang'] = [corpus.split('-')[1] for corpus in df['corpus']]
    print(df['lang'].value_counts())

    df['output_file_name'] = 'xling-' + df['corpus'] + '-' + df['conversation_id'] + '.parquet'

    groups = df.groupby(by=['output_file_name'])
    for out_f,dfi in groups:

        dfi[[col for col in list(dfi) if col != 'output_file_name']].to_parquet(
            os.path.join(OUTPUT_FOLDER, out_f[0]),
            engine='fastparquet',
            compression='snappy'
        )

  0%|          | 0/5 [00:00<?, ?it/s]

lang
cro    11756419
Name: count, dtype: int64
lang
spa    32093503
fra    11641191
ko      9170611
eng     6863556
zho     5244680
Name: count, dtype: int64
lang
fin    100099
Name: count, dtype: int64
lang
yid    21817
Name: count, dtype: int64
lang
eng    10444141
zho     6333029
deu     5473047
spa     3968514
Name: count, dtype: int64


## Sample/Test

In [ ]:
fs = grab_all_files(OUTPUT_FOLDER)
len(fs)

In [ ]:
for f in tqdm(np.random.choice(fs,size=(10,), replace=False)):
    df = pd.read_parquet(f)
    print(f.split('/')[-1])
    print(len(df))
    print(df.isna().sum())
    print(df['y_utterance'].sample(n=3).values.tolist())
    print('-----\n')

    # df.to_parquet(
    #     os.path.join('/Users/zacharyrosen/Desktop/', f.split('/')[-1]),
    #     engine='fastparquet',
    #     compression='snappy'
    # )
    # df.to_parquet(
    #     f,
    #     engine='fastparquet',
    #     compression='snappy'
    # )

## Sort and send

In [ ]:
fs = [f for f in os.listdir(OUTPUT_FOLDER) if not f.startswith('.') and (not os.path.isdir(os.path.join(OUTPUT_FOLDER,f)))]
len(fs)

In [ ]:
to_rosen = np.random.choice(fs, size=(int(len(fs)/2),), replace=False).tolist()
to_itkin = [f for f in fs if f not in to_rosen]
len(to_itkin), len(to_rosen)

In [ ]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

In [ ]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )

##### Sort and send (OLD)

In [ ]:
CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' in f) and (not f.startswith('.'))]
NON_CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' not in f) and (not f.startswith('.'))]

In [ ]:
to_rosen, to_itkin = [], []
to_rosen += np.random.choice(CANDOR, size=(int(len(CANDOR)/2),), replace=False).tolist() + np.random.choice(NON_CANDOR, size=(int(len(NON_CANDOR)/2),), replace=False).tolist()
to_itkin = [f for f in CANDOR if f not in to_rosen] + [f for f in NON_CANDOR if f not in to_rosen]
len(to_itkin), len(to_rosen)

In [ ]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

In [ ]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )